# **Space X Falcon 9 First Stage Landing Prediction**

## EDA with SQL

Estimated time needed: **60** minutes

### Objectives

- Understand the SpaceX dataset
- Load the dataset into a SQLite database
- Execute SQL queries to answer assignment questions

---

## Overview of the Dataset

SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars;
other providers cost upward of 165 million dollars each. Much of the savings is because
SpaceX can reuse the first stage.

If we can determine whether the first stage will land, we can determine the cost of a launch.
This information can be used if an alternate company wants to bid against SpaceX for a rocket launch.

## Setup — Load Data into SQLite

In [1]:
import csv
import sqlite3
import pandas as pd
import os

os.makedirs('../sql', exist_ok=True)
os.makedirs('../images', exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


In [2]:
# Create SQLite connection
con = sqlite3.connect("../data/spacex.db")
cur = con.cursor()

# Load the SpaceX CSV into the database
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")

# Remove blank rows
cur.execute("DROP TABLE IF EXISTS SPACEXTABLE")
cur.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")
con.commit()

print(f'Loaded {len(df)} rows into SPACEXTABLE')
print(f'Columns: {list(df.columns)}')

Loaded 101 rows into SPACEXTABLE
Columns: ['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


## Helper Function for SQL Queries

In [3]:
def run_query(query, description=""):
    """Execute a SQL query and return results as a DataFrame."""
    if description:
        print(f'--- {description} ---')
    result = pd.read_sql_query(query, con)
    print(result.to_string(index=False))
    print()
    return result

## Task 1: Display the Names of Unique Launch Sites

In [4]:
q1 = run_query("""
    SELECT DISTINCT "Launch_Site"
    FROM SPACEXTABLE
""", "Unique Launch Sites")

--- Unique Launch Sites ---
 Launch_Site
 CCAFS LC-40
 VAFB SLC-4E
  KSC LC-39A
CCAFS SLC-40



## Task 2: Display 5 Records Where Launch Sites Begin with 'CCA'

In [5]:
q2 = run_query("""
    SELECT *
    FROM SPACEXTABLE
    WHERE "Launch_Site" LIKE 'CCA%'
    LIMIT 5
""", "5 Records from CCA Sites")

--- 5 Records from CCA Sites ---
      Date Time (UTC) Booster_Version Launch_Site                                                       Payload  PAYLOAD_MASS__KG_     Orbit        Customer Mission_Outcome     Landing_Outcome
2010-06-04   18:45:00  F9 v1.0  B0003 CCAFS LC-40                          Dragon Spacecraft Qualification Unit                  0       LEO          SpaceX         Success Failure (parachute)
2010-12-08   15:43:00  F9 v1.0  B0004 CCAFS LC-40 Dragon demo flight C1, two CubeSats, barrel of Brouere cheese                  0 LEO (ISS) NASA (COTS) NRO         Success Failure (parachute)
2012-05-22    7:44:00  F9 v1.0  B0005 CCAFS LC-40                                         Dragon demo flight C2                525 LEO (ISS)     NASA (COTS)         Success          No attempt
2012-10-08    0:35:00  F9 v1.0  B0006 CCAFS LC-40                                                  SpaceX CRS-1                500 LEO (ISS)      NASA (CRS)         Success          No attempt
20

## Task 3: Display Total Payload Mass Carried by NASA (CRS)

In [6]:
q3 = run_query("""
    SELECT SUM("PAYLOAD_MASS__KG_") AS Total_Payload_Mass_kg
    FROM SPACEXTABLE
    WHERE "Customer" = 'NASA (CRS)'
""", "Total Payload Mass — NASA (CRS)")

--- Total Payload Mass — NASA (CRS) ---
 Total_Payload_Mass_kg
                 45596



## Task 4: Display Average Payload Mass for Booster Version F9 v1.1

In [7]:
q4 = run_query("""
    SELECT AVG("PAYLOAD_MASS__KG_") AS Avg_Payload_Mass_kg
    FROM SPACEXTABLE
    WHERE "Booster_Version" LIKE 'F9 v1.1%'
""", "Average Payload Mass — F9 v1.1")

--- Average Payload Mass — F9 v1.1 ---
 Avg_Payload_Mass_kg
         2534.666667



## Task 5: List the Date of the First Successful Landing on a Ground Pad

In [8]:
q5 = run_query("""
    SELECT MIN("Date") AS First_Successful_Ground_Landing
    FROM SPACEXTABLE
    WHERE "Landing_Outcome" = 'Success (ground pad)'
""", "First Successful Ground Pad Landing")

--- First Successful Ground Pad Landing ---
First_Successful_Ground_Landing
                     2015-12-22



## Task 6: List Boosters with Success in Drone Ship and Payload 4000–6000 kg

In [9]:
q6 = run_query("""
    SELECT "Booster_Version"
    FROM SPACEXTABLE
    WHERE "Landing_Outcome" = 'Success (drone ship)'
      AND "PAYLOAD_MASS__KG_" > 4000
      AND "PAYLOAD_MASS__KG_" < 6000
""", "Drone Ship Success, Payload 4000–6000 kg")

--- Drone Ship Success, Payload 4000–6000 kg ---
Booster_Version
    F9 FT B1022
    F9 FT B1026
 F9 FT  B1021.2
 F9 FT  B1031.2



## Task 7: List Total Number of Successful and Failed Mission Outcomes

In [10]:
q7 = run_query("""
    SELECT "Mission_Outcome", COUNT(*) AS Count
    FROM SPACEXTABLE
    GROUP BY "Mission_Outcome"
    ORDER BY Count DESC
""", "Mission Outcome Counts")

--- Mission Outcome Counts ---
                 Mission_Outcome  Count
                         Success     98
Success (payload status unclear)      1
                        Success       1
             Failure (in flight)      1



## Task 8: List Booster Versions That Carried the Maximum Payload Mass

In [11]:
q8 = run_query("""
    SELECT "Booster_Version"
    FROM SPACEXTABLE
    WHERE "PAYLOAD_MASS__KG_" = (
        SELECT MAX("PAYLOAD_MASS__KG_") FROM SPACEXTABLE
    )
""", "Boosters with Maximum Payload Mass")

--- Boosters with Maximum Payload Mass ---
Booster_Version
  F9 B5 B1048.4
  F9 B5 B1049.4
  F9 B5 B1051.3
  F9 B5 B1056.4
  F9 B5 B1048.5
  F9 B5 B1051.4
  F9 B5 B1049.5
 F9 B5 B1060.2 
 F9 B5 B1058.3 
  F9 B5 B1051.6
  F9 B5 B1060.3
 F9 B5 B1049.7 



## Task 9: List 2015 Failed Drone Ship Landings — Month, Booster, Launch Site

In [12]:
q9 = run_query("""
    SELECT substr("Date", 6, 2) AS Month,
           "Landing_Outcome",
           "Booster_Version",
           "Launch_Site"
    FROM SPACEXTABLE
    WHERE "Landing_Outcome" = 'Failure (drone ship)'
      AND substr("Date", 0, 5) = '2015'
""", "2015 Failed Drone Ship Landings")

--- 2015 Failed Drone Ship Landings ---


Month      Landing_Outcome Booster_Version Launch_Site
   01 Failure (drone ship)   F9 v1.1 B1012 CCAFS LC-40
   04 Failure (drone ship)   F9 v1.1 B1015 CCAFS LC-40



## Task 10: Rank Landing Outcomes Between 2010-06-04 and 2017-03-20

In [13]:
q10 = run_query("""
    SELECT "Landing_Outcome", COUNT(*) AS Count
    FROM SPACEXTABLE
    WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
    GROUP BY "Landing_Outcome"
    ORDER BY Count DESC
""", "Landing Outcome Rankings (2010-06-04 to 2017-03-20)")

--- Landing Outcome Rankings (2010-06-04 to 2017-03-20) ---
       Landing_Outcome  Count
            No attempt     10
  Success (drone ship)      5
  Failure (drone ship)      5
  Success (ground pad)      3
    Controlled (ocean)      3
  Uncontrolled (ocean)      2
   Failure (parachute)      2
Precluded (drone ship)      1



## Export SQL Scripts

Save all queries as a standalone `.sql` file for the repository.

In [14]:
sql_script = """-- SpaceX Falcon 9 SQL Analysis
-- All queries run against SPACEXTABLE in SQLite

-- Task 1: Unique launch sites
SELECT DISTINCT "Launch_Site" FROM SPACEXTABLE;

-- Task 2: 5 records from CCA sites
SELECT * FROM SPACEXTABLE WHERE "Launch_Site" LIKE 'CCA%' LIMIT 5;

-- Task 3: Total payload mass for NASA (CRS)
SELECT SUM("PAYLOAD_MASS__KG_") AS Total_Payload_Mass_kg
FROM SPACEXTABLE WHERE "Customer" = 'NASA (CRS)';

-- Task 4: Average payload mass for F9 v1.1
SELECT AVG("PAYLOAD_MASS__KG_") AS Avg_Payload_Mass_kg
FROM SPACEXTABLE WHERE "Booster_Version" LIKE 'F9 v1.1%';

-- Task 5: First successful ground pad landing
SELECT MIN("Date") AS First_Successful_Ground_Landing
FROM SPACEXTABLE WHERE "Landing_Outcome" = 'Success (ground pad)';

-- Task 6: Drone ship successes with payload 4000-6000 kg
SELECT "Booster_Version" FROM SPACEXTABLE
WHERE "Landing_Outcome" = 'Success (drone ship)'
  AND "PAYLOAD_MASS__KG_" > 4000 AND "PAYLOAD_MASS__KG_" < 6000;

-- Task 7: Mission outcome counts
SELECT "Mission_Outcome", COUNT(*) AS Count
FROM SPACEXTABLE GROUP BY "Mission_Outcome" ORDER BY Count DESC;

-- Task 8: Boosters with maximum payload mass
SELECT "Booster_Version" FROM SPACEXTABLE
WHERE "PAYLOAD_MASS__KG_" = (SELECT MAX("PAYLOAD_MASS__KG_") FROM SPACEXTABLE);

-- Task 9: 2015 failed drone ship landings
SELECT substr("Date", 6, 2) AS Month, "Landing_Outcome",
       "Booster_Version", "Launch_Site"
FROM SPACEXTABLE
WHERE "Landing_Outcome" = 'Failure (drone ship)'
  AND substr("Date", 0, 5) = '2015';

-- Task 10: Landing outcome rankings (2010-06-04 to 2017-03-20)
SELECT "Landing_Outcome", COUNT(*) AS Count
FROM SPACEXTABLE
WHERE "Date" BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY "Landing_Outcome" ORDER BY Count DESC;
"""

with open('../sql/spacex_queries.sql', 'w') as f:
    f.write(sql_script)

print('SQL script saved to sql/spacex_queries.sql')

SQL script saved to sql/spacex_queries.sql


In [15]:
# Close the database connection
con.close()
print('Database connection closed.')

Database connection closed.


## Summary

In this notebook we:

1. **Loaded** the SpaceX dataset into a SQLite database
2. **Executed** 10 SQL queries covering:
   - Unique launch sites
   - Site-specific records (CCA)
   - Payload mass aggregations (total and average)
   - First successful ground pad landing date
   - Drone ship success with payload filters
   - Mission outcome tallies
   - Maximum payload booster versions
   - Time-filtered failure analysis (2015)
   - Landing outcome rankings over a date range
3. **Exported** all queries as a standalone SQL script